In [1]:
from pathlib import Path
import re
import subprocess
import shlex
from datetime import datetime

# Where your run folders live (adjust if needed)
RUNS_ROOT = Path("/home/jovyan/work-easi-eds/nvms_runs/run1")   # <-- change if yours differs

PIPELINE = Path("/home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py")
TILE_SHP = Path("/home/jovyan/assets/eds_lsat_grid_min_max.shp")

SPAN_YEARS = 10

# Toggle these
DRY_RUN = False          # matches your normal command
RUN_DOWNLOAD = True     # you said: pull the data for each run

# Where the pipeline typically writes downloaded tiles (adjust if your pipeline uses a different root)
# If you're unsure, keep the marker-only skip (still works).
DOWNLOAD_ROOT = Path("/home/jovyan/scratch/eds/tiles")

print("RUNS_ROOT:", RUNS_ROOT)
print("PIPELINE:", PIPELINE)
print("TILE_SHP:", TILE_SHP)
print("DOWNLOAD_ROOT:", DOWNLOAD_ROOT)


RUNS_ROOT: /home/jovyan/work-easi-eds/nvms_runs/run1
PIPELINE: /home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py
TILE_SHP: /home/jovyan/assets/eds_lsat_grid_min_max.shp
DOWNLOAD_ROOT: /home/jovyan/scratch/eds/tiles


In [2]:
RUN_RE = re.compile(
    r"""
    _(?P<scene>p\d{3}r\d{3})      # p089r078
    _d(?P<start>\d{8})(?P<end>\d{8})  # dYYYYMMDDYYYYMMDD
    """,
    re.VERBOSE | re.IGNORECASE,
)

def parse_run_folder(folder: Path):
    m = RUN_RE.search(folder.name)
    if not m:
        return None

    scene = m.group("scene").lower()          # p089r078
    start = m.group("start")                  # 20230306
    end = m.group("end")                      # 20231024

    # pipeline expects --tile-id p095r081 style
    tile_id = scene

    # your CLI example uses YYYY-MM-DD for download start/end
    start_iso = datetime.strptime(start, "%Y%m%d").strftime("%Y-%m-%d")
    end_iso = datetime.strptime(end, "%Y%m%d").strftime("%Y-%m-%d")

    return {
        "run_dir": folder,
        "tile_id": tile_id,
        "start": start,
        "end": end,
        "start_iso": start_iso,
        "end_iso": end_iso,
    }

# quick test: show what we detect
runs = []
for p in sorted(RUNS_ROOT.iterdir()):
    if p.is_dir():
        info = parse_run_folder(p)
        if info:
            runs.append(info)

print(f"Detected {len(runs)} runnable folders:")
for r in runs:
    print(" -", r["run_dir"].name, "=>", r["tile_id"], r["start_iso"], "to", r["end_iso"])


Detected 145 runnable folders:
 - lzolre_p089r078_d2023030620231024_dlwm6 => p089r078 2023-03-06 to 2023-10-24
 - lzolre_p089r084_d2023012520231024_dlwm6 => p089r084 2023-01-25 to 2023-10-24
 - lzolre_p090r088_d2023010820231015_dlwm5 => p090r088 2023-01-08 to 2023-10-15
 - lzolre_p090r090_d2023010820230727_dlwm5 => p090r090 2023-01-08 to 2023-07-27
 - lzolre_p091r076_d2023030420231022_dlwm6 => p091r076 2023-03-04 to 2023-10-22
 - lzolre_p091r087_d2023030420230928_dlwm5 => p091r087 2023-03-04 to 2023-09-28
 - lzolre_p091r089_d2023011520230216_dlwm5 => p091r089 2023-01-15 to 2023-02-16
 - lzolre_p091r090_d2023011520230904_dlwm5 => p091r090 2023-01-15 to 2023-09-04
 - lzolre_p092r075_d2023041220231029_dlwm6 => p092r075 2023-04-12 to 2023-10-29
 - lzolre_p092r084_d2023022420230928_dlwm5 => p092r084 2023-02-24 to 2023-09-28
 - lzolre_p092r085_d2023022420231006_dlwm5 => p092r085 2023-02-24 to 2023-10-06
 - lzolre_p092r086_d2023022420230928_dlwm5 => p092r086 2023-02-24 to 2023-09-28
 - lzolre

In [3]:
def done_marker(run_dir: Path) -> Path:
    return run_dir / ".download.done"

def looks_downloaded(tile_id: str, start: str, end: str) -> bool:
    """
    Best-effort check: does DOWNLOAD_ROOT/<tile_id>/sr/... contain either date?
    This avoids rerunning downloads if data already exists.

    If your pipeline writes somewhere else, either:
      - change DOWNLOAD_ROOT, or
      - rely on the .download.done marker only.
    """
    scene_dir = DOWNLOAD_ROOT / tile_id
    if not scene_dir.exists():
        return False

    # Search for date strings in filenames under sr and fc (cheap-ish glob)
    # Keep it simple and robust.
    for sub in ("sr", "fc"):
        base = scene_dir / sub
        if not base.exists():
            continue

        # recursive glob for any tif containing the date
        if list(base.rglob(f"*{start}*.tif")):
            return True
        if list(base.rglob(f"*{end}*.tif")):
            return True

    return False


In [4]:
from pathlib import Path
import pandas as pd
import re

DOWNLOAD_ROOT = Path("/home/jovyan/scratch/eds/tiles")  # adjust if needed

print("Scanning:", DOWNLOAD_ROOT)


Scanning: /home/jovyan/scratch/eds/tiles


In [5]:
def count_tiles(tile_dir: Path, subdir: str) -> int:
    d = tile_dir / subdir
    if not d.exists():
        return 0
    return len(list(d.rglob("*.tif")))


In [6]:
rows = []

for tile_dir in sorted(DOWNLOAD_ROOT.iterdir()):
    if not tile_dir.is_dir():
        continue

    tile_id = tile_dir.name.lower()

    sr_count = count_tiles(tile_dir, "sr")
    fc_count = count_tiles(tile_dir, "fc")
    fmask_count = count_tiles(tile_dir, "fmask")

    rows.append({
        "tile_id": tile_id,
        "sr_tiles": sr_count,
        "fc_tiles": fc_count,
        "fmask_tiles": fmask_count,
        "has_fmask": fmask_count > 0,
    })

df = pd.DataFrame(rows).sort_values("tile_id").reset_index(drop=True)
df


,tile_id,sr_tiles,fc_tiles,fmask_tiles,has_fmask
0,p089r078,378,378,0,False
1,p089r084,290,290,0,False
2,p090r084,276,276,0,False
3,p090r088,26,26,0,False
4,p090r090,10,10,0,False
5,p091r076,46,46,0,False
6,p091r087,20,20,0,False
7,p091r089,6,6,0,False
8,p091r090,24,24,0,False
9,p092r075,42,42,0,False


### -----------------------  DELETE specific or empty dirs ------------

In [ ]:
tiles_to_delete = df[
    (df.sr_tiles <= 50) & (df.fc_tiles <= 50)
]["tile_id"].tolist()

tiles_to_delete


In [ ]:
# ## manually specify tiles

# tiles_to_delete = [
#     "p090r078",
#     "p090r088",
#     "p091r084",
#     "p093r084",
# ]


In [ ]:
from pathlib import Path

DOWNLOAD_ROOT = Path("/home/jovyan/scratch/eds/tiles")
RUNS_ROOT = Path("/home/jovyan/work-easi-eds/nvms_runs/run1")

print("Tile data directories that would be deleted:\n")

for tile_id in tiles_to_delete:
    tile_path = DOWNLOAD_ROOT / tile_id
    if tile_path.exists():
        print("  DATA:", tile_path)
    else:
        print("  DATA: (missing)", tile_path)

print("\nRun marker files that would be deleted:\n")

for run_dir in RUNS_ROOT.iterdir():
    if not run_dir.is_dir():
        continue

    for tile_id in tiles_to_delete:
        if tile_id in run_dir.name:
            for marker in [".download.done", ".download.failed"]:
                m = run_dir / marker
                if m.exists():
                    print("  MARKER:", m)


In [ ]:
import shutil

confirm = input(
    "⚠️  This will permanently delete selected tile datasets AND run markers.\n"
    "Type YES to continue: "
)

if confirm != "YES":
    raise SystemExit("Aborted by user")

# Delete tile data
for tile_id in tiles_to_delete:
    tile_path = DOWNLOAD_ROOT / tile_id
    if tile_path.exists():
        print("Deleting tile data:", tile_path)
        shutil.rmtree(tile_path)
    else:
        print("Tile data already missing:", tile_path)

# Delete stale markers
for run_dir in RUNS_ROOT.iterdir():
    if not run_dir.is_dir():
        continue

    for tile_id in tiles_to_delete:
        if tile_id in run_dir.name:
            for marker in [".download.done", ".download.failed"]:
                m = run_dir / marker
                if m.exists():
                    print("Deleting marker:", m)
                    m.unlink()

print("\nCleanup complete.")


## -------------------------------- Pull the trigger ----------------------------------

In [ ]:
def build_cmd(tile_id: str, start_iso: str, end_iso: str) -> list[str]:
    cmd = [
        "python",
        str(PIPELINE),
        "--tile-shp", str(TILE_SHP),
        "--tile-id", tile_id,
        "--span-years", str(SPAN_YEARS),
    ]

    if RUN_DOWNLOAD:
        cmd.append("--run-download")

    # cmd += [
    #     "--download-start-date", start_iso,
    #     "--download-end-date", end_iso,
    # ]

    def yyyymm(iso_date: str) -> str:
        return iso_date.replace("-", "")[:6]

    cmd += [
        "--season-core-start", yyyymm(start_iso),
        "--season-core-end",   yyyymm(end_iso),
    ]

    if DRY_RUN:
        cmd.append("--dry-run")

    return cmd


ok = 0
skipped = 0
failed = 0

# Run on multiple tiles (2) at a time

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import subprocess, time, os, shutil

MAX_WORKERS = 2  # start safe
LOCK_ROOT = Path("/home/jovyan/scratch/eds/.tile_locks")
LOCK_ROOT.mkdir(parents=True, exist_ok=True)

def acquire_lock(lock_dir: Path, timeout_s: int = 6 * 60 * 60, poll_s: float = 2.0) -> None:
    start = time.time()
    while True:
        try:
            lock_dir.mkdir(parents=True, exist_ok=False)  # atomic
            (lock_dir / "pid.txt").write_text(str(os.getpid()))
            return
        except FileExistsError:
            if time.time() - start > timeout_s:
                raise TimeoutError(f"Timed out waiting for lock: {lock_dir}")
            time.sleep(poll_s)

def release_lock(lock_dir: Path) -> None:
    if lock_dir.exists():
        shutil.rmtree(lock_dir, ignore_errors=True)

def run_one_job(cmd, run_dir_str, tile_id, start, end, download_root_str):
    run_dir = Path(run_dir_str)
    download_root = Path(download_root_str)

    marker_done = run_dir / ".download.done"
    marker_fail = run_dir / ".download.failed"
    lock_dir = LOCK_ROOT / tile_id

    def looks_downloaded_local():
        scene_dir = download_root / tile_id
        if not scene_dir.exists():
            return False
        for sub in ("sr", "fc"):
            base = scene_dir / sub
            if not base.exists():
                continue
            if list(base.rglob(f"*{start}*.tif")) or list(base.rglob(f"*{end}*.tif")):
                return True
        return False

    # marker only counts if data exists (prevents stale-marker skipping)
    if marker_done.exists() and looks_downloaded_local():
        return ("skipped", run_dir.name, "marker+data")

    if looks_downloaded_local():
        marker_done.write_text("skipped: already present\n")
        return ("skipped", run_dir.name, "data exists")

    acquire_lock(lock_dir)
    try:
        proc = subprocess.run(cmd, text=True, capture_output=True)
        if proc.returncode != 0:
            marker_fail.write_text(proc.stderr + "\n")
            return ("failed", run_dir.name, proc.stderr[-4000:])
        marker_done.write_text("ok\n")
        return ("ok", run_dir.name, proc.stdout[-2000:])
    finally:
        release_lock(lock_dir)

# Build jobs from your runs
jobs = []
for r in runs:
    cmd = build_cmd(r["tile_id"], r["start_iso"], r["end_iso"])
    jobs.append((cmd, str(r["run_dir"]), r["tile_id"], r["start"], r["end"], str(DOWNLOAD_ROOT)))

print(f"Prepared {len(jobs)} jobs. Running with MAX_WORKERS={MAX_WORKERS} (tile-locked).")

ok = skipped = failed = 0

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(run_one_job, *job) for job in jobs]
    for fut in as_completed(futures):
        status, name, msg = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skipped":
            skipped += 1
        else:
            failed += 1
        print(f"[{status.upper():7}] {name} — {msg}")

print(f"\nDone. ok={ok}, skipped={skipped}, failed={failed}")


## Run on one tile at a time

In [ ]:


for r in runs:
    run_dir = r["run_dir"]
    tile_id = r["tile_id"]
    start = r["start"]
    end = r["end"]

    marker = done_marker(run_dir)

    # Skip rules
    if marker.exists():
        print(f"[SKIP] {run_dir.name} (marker exists)")
        skipped += 1
        continue

    if looks_downloaded(tile_id, start, end):
        print(f"[SKIP] {run_dir.name} (looks already downloaded under {DOWNLOAD_ROOT}/{tile_id})")
        # write marker so we don't keep rechecking
        marker.write_text("skipped: already present\n")
        skipped += 1
        continue

    cmd = build_cmd(tile_id, r["start_iso"], r["end_iso"])
    print("\n[RUN ]", run_dir.name)
    print("      ", " ".join(shlex.quote(c) for c in cmd))

    try:
        # run it (captures output so notebook doesn't explode)
        proc = subprocess.run(cmd, text=True, capture_output=True)
        print(proc.stdout)
        if proc.returncode != 0:
            print(proc.stderr)
            raise RuntimeError(f"returncode={proc.returncode}")

        # mark success
        marker.write_text("ok\n")
        ok += 1

    except Exception as e:
        failed += 1
        (run_dir / ".download.failed").write_text(str(e) + "\n")
        print(f"[FAIL] {run_dir.name}: {e}")

print(f"\nDone. ok={ok}, skipped={skipped}, failed={failed}")
